# nls_mpc

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/nls_mpc.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们使用 CFPS 家庭经济数据，先估计最简单的线性消费函数：消费对家庭收入做 OLS 回归，并保存线性拟合值作为对照。线性设定隐含边际消费倾向（MPC）为常数。

In [ ]:
%%stata
clear
set more off
use "../datasets/cfps_family_econ.dta"
// 线性回归
reg expense fincome1
predict expense_linear

接着清洗数据：剔除收入、消费的缺失值与负值，并剔除消费超过收入 5 倍的极端观测，然后用 nl 做非线性最小二乘，估计 Keynes 型消费函数 expense = α + β·income^γ。花括号中 {gamma=1} 给出参数初值；当 γ<1 时消费函数为凹函数，MPC 随收入递减，这是非线性设定要捕捉的经济特征。

In [ ]:
%%stata
// 非线性回归
drop if fincome1==. | fincome1<0
drop if expense==. |expense<0 | expense>5*fincome1
nl (expense={alpha=_b[_cons]}+{beta=_b[fincome1]}*fincome1^{gamma=1})
predict expense_nls

由估计的消费函数直接计算每个收入水平上的边际消费倾向：对 β·income^γ 求导得 MPC = β·γ·income^(γ-1)。这里 _b[/beta] 等引用 nl 方程参数；注意这是样本内的 MPC 函数，随收入水平变化。

In [ ]:
%%stata
// 计算MPC
gen mpc=_b[/beta]*_b[/gamma]*fincome1^(_b[/gamma]-1)

最后按收入排序，把消费散点与线性、非线性两条拟合曲线叠加绘图，直观比较两种设定下拟合曲线形状的差异。

In [ ]:
%%stata
// 画图
sort fincome1
twoway (scatter expense fincome1) (line expense_linear fincome1) (line expense_nls fincome1)